In [1]:
import requests
import pandas as pd
import numpy as np
import polars as pl

In [2]:
# Fetch help contacts data
help_contacts = requests.get("http://api.justgood.work/api/v2/help-contacts")
help_contacts_json = help_contacts.json()["data"]
help_contacts_json

[{'_id': '650440891b0bd438ce98c950',
  'deviceId': 'jgw-vanilla-9d559236d8df7df0-KAZ -> GBR',
  'appId': 'jgw-vanilla',
  'locale': 'KAZ',
  'route': 'KAZ -> GBR',
  'phoneNumber': '0300 123 1100',
  'organization': 'Seasonal Worker Scheme - Generic',
  'email': '',
  'createdAt': '2023-09-15T11:31:21.549Z',
  'updatedAt': '2023-09-15T11:31:21.549Z',
  '__v': 0},
 {'_id': '64f5304d636a8f3d130d4c34',
  'deviceId': 'jgw-vanilla-b17966807350e882-GBR -> GBR',
  'appId': 'jgw-vanilla',
  'locale': 'GBR',
  'route': 'GBR -> GBR',
  'phoneNumber': '0300 400 100',
  'organization': '',
  'email': '',
  'createdAt': '2023-09-04T01:18:05.942Z',
  'updatedAt': '2023-09-04T01:18:05.942Z',
  '__v': 0},
 {'_id': '64e905053823dac17a4d829d',
  'deviceId': 'jgw-vanilla-9BE1DE30-5A0C-4F99-B489-1D36EE8D126F-UZB -> GBR',
  'appId': 'jgw-vanilla',
  'locale': 'UZB',
  'route': 'UZB -> GBR',
  'phoneNumber': '0808 800 0082',
  'organization': 'Seasonal Worker Scheme - Pro-Force',
  'email': '',
  'createdAt

In [8]:
# Convert help contacts json to a pandas Dataframe
hc_df = pd.DataFrame.from_records(help_contacts_json)

# view the first 2 records
hc_df.columns

Index(['_id', 'deviceId', 'appId', 'locale', 'route', 'phoneNumber',
       'organization', 'email', 'createdAt', 'updatedAt', '__v'],
      dtype='object')

In [4]:
# generics
def count_unique(df, field):
    return df[field].nunique()

In [5]:
start_date = "2023-05-01" 
end_date = "2023-05-31"
mask = hc_df["createdAt"].between(start_date, end_date, inclusive="both")
hc_df = hc_df.loc[mask]

In [6]:
# phone numbers and emails by route    
def df_aggregate_by_group(df, result_fields, group_field):
    df[group_field] = df[group_field].replace("", "Not specified")
    
    aggregation_dict = {field: lambda values: values.count() for field in result_fields}
    grouped_data = df.groupby(group_field)[result_fields].agg(aggregation_dict)
    
    result = {
        "labels": grouped_data.index.values,
    }
    
    for field in result_fields:
        result[field] = grouped_data[field].values
    
    return result

In [7]:
def dictionary_from_two_lists(labels, values):
    """Creates a dictionary from two lists

    Args:
        labels (_list_): to be used as keys
        values (_list_): to be used as values

    Returns:
        _type_: returns a dictionary 
    """
    new_dict = {}

    for label, value in zip(labels, values):
        new_dict[label] = value

    return new_dict


_df = hc_df
emails_and_phonenumbers_clicks_by_route = df_aggregate_by_group(
    _df,
    result_fields=["phoneNumber", "email"],
    group_field="route"
)
emails_and_phonenumbers_clicks_by_org = df_aggregate_by_group(
    _df,
    result_fields=["phoneNumber", "email"],
    group_field="organization"
)
phone_numbers_by_route = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_route.get("labels"),
    emails_and_phonenumbers_clicks_by_route.get("phoneNumber")
)
emails_by_route = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_route.get("labels"),
    emails_and_phonenumbers_clicks_by_route.get("email")
)

phone_numbers_by_org = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_org.get("labels"),
    emails_and_phonenumbers_clicks_by_org.get("phoneNumber")
)
emails_by_org = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_org.get("labels"),
    emails_and_phonenumbers_clicks_by_org.get("email")
)
print({
    "phone numbers clicks by route": phone_numbers_by_route, 
    "emails clicks by route": emails_by_route,
    # "phone_numbers_by_org":phone_numbers_by_org,
    # "emails_by_org":emails_by_org
})
phone_numbers_by_route, emails_by_route

{'phone numbers clicks by route': {}, 'emails clicks by route': {}}


({}, {})